# 02 – Zero-Shot Vision-Language Models

**Phase 3 exit gate:** Best zero-shot config chosen on validation. Features cached for top 2–3 models.

**Needs GPU** for feature extraction. Everything after loading cached features runs on CPU.

**Deliverables:** Cached `.npy` feature files, comparison table of models × prompt sets, this notebook.

In [ ]:
import sys, pathlib, json
import numpy as np
import torch

sys.path.insert(0, "..")
sys.path.append("/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

from src.config import DATA_DIR, OUT_DIR, seed_everything, load_config
seed_everything(42)

import torchvision, timm, sklearn
try:
    import open_clip
    print("open_clip version:", open_clip.__version__)
except ImportError:
    print("WARNING: open_clip not installed – run: pip install open-clip-torch")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import pandas as pd
from src.data import load_csvs, GalleryDataset, image_dir, parse_val_ground_truth

dfs = load_csvs(DATA_DIR)
train_df    = dfs["train"]
val_gallery = dfs["val_gallery"]
val_queries = dfs["val_queries"]
test_gallery= dfs["test_gallery"]
test_queries= dfs["test_queries"]
meta        = dfs["category_metadata"]
val_gt      = parse_val_ground_truth(dfs["val_gt"])

## 1. Extract image features (per model)

In [ ]:
from torch.utils.data import DataLoader
from src.models import load_clip_model
from src.features import extract_image_features, save_features, load_features

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Models to evaluate – subset for quick iteration; expand as needed
MODELS = [
    {"name": "ViT-L-14",            "pretrained": "openai",            "tag": "vitl14_openai"},
    {"name": "ViT-SO400M-14-SigLIP","pretrained": "webli",             "tag": "siglip_so400m"},
    # {"name": "ViT-H-14",          "pretrained": "laion2b_s32b_b79k", "tag": "vith14_laion2b"},
]

for m in MODELS:
    print(f"\n=== {m['tag']} ===")
    feat_dir = pathlib.Path(OUT_DIR) / "features" / m["tag"]

    model, preprocess, _ = load_clip_model(m["name"], m["pretrained"], DEVICE)

    for split, df in [("train", train_df), ("val", val_gallery), ("test", test_gallery)]:
        cache = feat_dir / f"{split}_image.npy"
        if cache.exists():
            print(f"  {split}: cache hit, skipping.")
            continue
        img_dir = image_dir(DATA_DIR, split if split != "val" else "val")
        ds = GalleryDataset(df, img_dir, transform=preprocess)
        loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
        feats, ids = extract_image_features(model, loader, device=DEVICE, use_tta=True)
        save_features(feats, ids, feat_dir, split)

    del model
    torch.cuda.empty_cache()

## 2. Build category text features (prompt ensembling)

In [ ]:
from src.features import build_prompt_texts, aggregate_text_features, save_text_features
from src.text_matching import build_surface_form_dict
import ast

cat_names    = meta["category_name"].tolist()
n_categories = len(cat_names)

# Build alternative names dict {idx: [alt_name, ...]}
alt_names_dict = {}
for idx, row in meta.reset_index(drop=True).iterrows():
    raw = row.get("alternative_names", "")
    if isinstance(raw, str) and raw:
        try:
            alt_names_dict[idx] = ast.literal_eval(raw)
        except Exception:
            alt_names_dict[idx] = [s.strip() for s in raw.split(",") if s.strip()]

desc_dict = {idx: str(row["description"]) for idx, row in meta.reset_index(drop=True).iterrows()}

texts, cat_indices = build_prompt_texts(
    cat_names,
    alternative_names=alt_names_dict,
    descriptions=desc_dict,
)
print(f"Total prompts: {len(texts)} for {n_categories} categories")

In [ ]:
from src.models import load_clip_model
from src.features import extract_text_features, aggregate_text_features, save_text_features

for m in MODELS:
    print(f"\n=== text features: {m['tag']} ===")
    feat_dir = pathlib.Path(OUT_DIR) / "features" / m["tag"]
    text_cache = feat_dir / "text.npy"
    if text_cache.exists():
        print("  text cache hit, skipping.")
        continue

    model, _, tokenize = load_clip_model(m["name"], m["pretrained"], DEVICE)

    def encode_texts(batch):
        tokens = tokenize(batch).to(DEVICE)
        with torch.no_grad():
            emb = model.encode_text(tokens)
        return emb.float().cpu().numpy()

    raw_text_feats = extract_text_features(encode_texts, texts)
    cat_proto = aggregate_text_features(raw_text_feats, cat_indices, n_categories)
    save_text_features(cat_proto, list(range(n_categories)), feat_dir, tag="text")

    del model
    torch.cuda.empty_cache()

## 3. Zero-shot scoring & validation

In [ ]:
from src.features import load_features, load_text_features
from src.text_matching import TextMatcher, build_surface_form_dict
from src.metrics import evaluate, print_metrics
from src.ensemble import temperature_scale

surface_forms = build_surface_form_dict(meta)
matcher = TextMatcher(surface_forms, meta_df=meta)

# category_id → 0-based index (from meta DataFrame order)
cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {int(c): i for i, c in enumerate(cat_id_list)}

results_table = []

for m in MODELS:
    feat_dir = pathlib.Path(OUT_DIR) / "features" / m["tag"]
    X_val, val_ids = load_features(feat_dir, "val", gallery_df=val_gallery)
    T_val, _       = load_text_features(feat_dir, tag="text")

    # Raw cosine scores: (N_val, K)
    S_raw = X_val @ T_val.T

    # Softmax-over-categories (recommended)
    S_soft = temperature_scale(S_raw, T=1.0, to_prob=True)

    pred_dict = {}
    for _, row in val_queries.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order = np.argsort(-S_soft[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [val_ids[j] for j in order[:20]]

    metrics = evaluate(pred_dict, val_gt)
    results_table.append({"model": m["tag"], **metrics})
    print(f"\n{m['tag']}:")
    print_metrics(metrics)

import pandas as pd
print("\n--- Summary ---")
print(pd.DataFrame(results_table).to_string(index=False))

## 4. Temperature tuning (on train OOF or val)

In [ ]:
# TODO: use fit_temperature() from src/ensemble.py once train OOF logits are available
# from src.ensemble import fit_temperature
# T_opt = fit_temperature(S_train_logits, y_train_labels)
print("Temperature tuning: run after Phase 4 when OOF logits are available.")